In [1]:
from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import OllamaEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_community.cross_encoders import HuggingFaceCrossEncoder

from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

from langchain_classic.retrievers import ContextualCompressionRetriever

C:\Users\gagan\AppData\Local\Temp\ipykernel_59396\3221271015.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = """
Doc 1:
Python is a great programming language
for AI and machine learning.

Doc 2:
The python is a large snake
native to tropical regions.

Doc 3:
The company reported
a 20% increase in revenue.

Doc 4:
To learn Python,
start with basic syntax
and loops.

Doc 5:
Machine learning models
require clean data.

Doc 6:
I saw a huge python
in the zoo yesterday.
"""

with open(
    "python_data.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(data)

In [3]:
loader = TextLoader("python_data.txt")

docs = loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0
)

splits = splitter.split_documents(docs)

print(len(splits))

6


In [5]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [6]:
vectorstore = FAISS.from_documents(
    splits,
    embeddings
)

In [7]:
base_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

In [8]:
model_name = "BAAI/bge-reranker-base"

cross_encoder = HuggingFaceCrossEncoder(
    model_name=model_name
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12635.34it/s]


In [9]:
compressor = CrossEncoderReranker(
    model=cross_encoder,
    top_n=2
)

In [10]:
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [11]:
query = "What is a python and where does it live?"

In [12]:
print("=" * 80)
print("BASE RETRIEVER")
print("=" * 80)

docs = base_retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"\nRank {i}\n")
    print(doc.page_content)

BASE RETRIEVER

Rank 1

Doc 2:
The python is a large snake
native to tropical regions.

Rank 2

Doc 6:
I saw a huge python
in the zoo yesterday.

Rank 3

Doc 1:
Python is a great programming language
for AI and machine learning.

Rank 4

Doc 4:
To learn Python,
start with basic syntax
and loops.

Rank 5

Doc 5:
Machine learning models
require clean data.


In [13]:
print("=" * 80)
print("RERANKED")
print("=" * 80)

docs = rerank_retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"\nRank {i}\n")
    print(doc.page_content)

RERANKED

Rank 1

Doc 1:
Python is a great programming language
for AI and machine learning.

Rank 2

Doc 2:
The python is a large snake
native to tropical regions.
